In [3]:
# install packages in colab if necessary
#!pip install nlpaug tdqm sacremoses

import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
import nlpaug.augmenter.sentence as nas
import nlpaug.flow as nafc
from transformers import MarianMTModel, MarianTokenizer
import numpy as np
import pandas as pd
import torch
import gc
from tqdm import tqdm
import re

tqdm.pandas()


rseed = 42

In [4]:
from huggingface_hub import login
from getpass import getpass

hftoken = getpass("HF-Token ")
login(token=hftoken)

KeyboardInterrupt: Interrupted by user

In [ ]:
# Synonym Augmenter
# prob 0.4
def synonymizer(text, aug1):
    augmented_text = aug1.augment(text)
    augmented_text = augmented_text[0] if isinstance(augmented_text, list) else augmented_text
    return augmented_text
#    else:
#        augmented_text = aug2.augment(text)
#        return augmented_text[0] if isinstance(augmented_text, list) else augmented_text

In [ ]:
# random swap
# prob 0.05
def random_swap(text, aug):
    augmented_text = aug.augment(text)
    return augmented_text[0] if isinstance(augmented_text, list) else augmented_text


#  random deletion
# prob 0.05
def random_del(text, aug):
    augmented_text = aug.augment(text)
    return augmented_text[0] if isinstance(augmented_text, list) else augmented_text

In [ ]:
# data local
#df = pd.read_csv("..\data\csv\mbti_cleaned.csv")

# data colab
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv("/content/drive/MyDrive/Data/mbti_cleaned.csv")

#data alvis
syn_sample = pd.read_csv("~/MA/data/syn_sample.csv", sep = "\t", quoting=1)
sw_sample = pd.read_csv("~/MA/data/sw_sample.csv", sep = "\t", quoting=1)
del_sample = pd.read_csv("~/MA/data/del_sample.csv", sep = "\t", quoting=1)


<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Tim\AppData\Local\Temp\ipykernel_12148\1371491495.py:2: SyntaxWarning: invalid escape sequence '\d'
  df = pd.read_csv("..\data\csv\mbti_cleaned.csv")


In [ ]:
def data_augmenter(syn_sample, bt_sample, sw_sample, del_sample):
    #device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    
    #aug_syn_word = naw.SynonymAug(aug_src = "wordnet")
    aug_syn_bert =naw.ContextualWordEmbsAug(
        model_path='bert-base-uncased',
        action="substitute",
        device="cuda",
        stopwords_regex=r"<mbti>"
    )
    # can also add wordnet if wanted
    syn_sample["post_augmented"] = syn_sample["post"].progress_apply(synonymizer, args=(aug_syn_bert,))

    # free up vram
    #del aug_syn_word
    del aug_syn_bert

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


    aug_rand_swap = naw.RandomWordAug(action = "swap",
                                      stopwords_regex=r"<mbti>")
    sw_sample["post_augmented"] = sw_sample["post"].progress_apply(random_swap, args = (aug_rand_swap,))
    
    aug_rand_del = naw.RandomWordAug(action = "delete",
                                     stopwords_regex=r"<mbti>")
    del_sample["post_augmented"] = del_sample["post"].progress_apply(random_del, args = (aug_rand_del,))


    return syn_sample,sw_sample,del_sample



In [ ]:
# fixing nltk dependency
import nltk
#nltk.download('averaged_perceptron_tagger_eng')

# run data_augmenter
syn_df, sw_df, del_df = data_augmenter(syn_sample,sw_sample, del_sample)

In [2]:
#saving data
# syn_df.to_csv("syn_augmented_ver2.csv", sep='\t', index=False, quoting=1)
# sw_df.to_csv("sw_augmented_ver2.csv", sep='\t', index=False, quoting=1)
# del_df.to_csv("del_augmented_ver2.csv", sep='\t', index=False, quoting=1)
# print("Alle Dateien erfolgreich als CSV gespeichert!")


import pandas as pd
# reading finished datasets locally
syn_df = pd.read_csv("../data/csv/syn_augmented.csv", sep = "\t", quoting = 1)
bt_df = pd.read_csv("../data/csv/bt_augmented.csv", sep = "\t", quoting = 1)
sw_df = pd.read_csv("../data/csv/sw_augmented.csv", sep = "\t", quoting = 1)
del_df = pd.read_csv("../data/csv/del_augmented.csv", sep = "\t", quoting = 1)

# bt_df.head()

In [ ]:
# print(f"syn_df mbti mask counts in post col (pre aug): {syn_df["post"].apply(lambda x: "<mbti>" in x).sum()}")
# print(f"syn_df mbti mask counts in augmented col: {syn_df["post_augmented"].apply(lambda x: "<mbti>" in x).sum()}")

# print(f"sw_df mbti mask counts in post col (pre aug): {sw_df["post"].apply(lambda x: "<mbti>" in x).sum()}")
# print(f"sw_df mbti mask counts in augmented col: {sw_df["post_augmented"].apply(lambda x: "<mbti>" in x).sum()}")

# print(f"del_df mbti mask counts in post col (pre aug): {del_df["post"].apply(lambda x: "<mbti>" in x).sum()}")
# print(f"del_df mbti mask counts in augmented col: {del_df["post_augmented"].apply(lambda x: "<mbti>" in x).sum()}")

bt_df mbti mask counts in post col (pre aug): 8269
bt_df mbti mask counts in augmented col: 7042
bt_df failed mbti masking example, counts in augmented col: 19
syn_df mbti mask counts in post col (pre aug): 2398
syn_df mbti mask counts in augmented col: 1917
sw_df mbti mask counts in post col (pre aug): 574
sw_df mbti mask counts in augmented col: 285
del_df mbti mask counts in post col (pre aug): 575
del_df mbti mask counts in augmented col: 296
